## Section 5-6 experiment: MLMC vs MLQMC on exotic payoffs

This notebook implements the experiment discussed in Sections 5 and 6:

- Section 5: add **MLQMC** (randomized rank-1 lattice + Brownian bridge) to the multilevel framework,
- Section 6: compare **MLMC** vs **MLQMC** on exotic payoffs under the same GBM setting.

We keep the telescoping estimator
$$\mathbb{E}[\hat P_L]=\mathbb{E}[\hat P_0]+\sum_{l=1}^{L}\mathbb{E}[\hat P_l-\hat P_{l-1}],$$
and only change how level expectations are sampled (MC vs randomized QMC).

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

np.random.seed(2026)

IMAGES_DIR = "/Users/AnranSeverac/SimulationMethods/Figures"
os.makedirs(IMAGES_DIR, exist_ok=True)

# GBM experiment setup
r_ex = 0.05
sigma_ex = 0.2
T_ex = 1.0
S0_ex = 1.0
K_ex = 1.0
B_ex = 0.8
M_ex = 4

# Target RMSE ε: geometric grid from 2e-2 down to 2e-3 (8 values). Shorter run: slice, e.g. eps_grid_ex[::2].
eps_grid_ex = np.geomspace(0.02, 0.002, num=8)

### Discretisation schemes, payoff definitions, and path construction

We support two SDE discretisations for GBM $dS=rS\,dt+\sigma S\,dW$:

**Euler–Maruyama** (strong order 0.5):
$$S_{i+1}=S_i + rS_i h + \sigma S_i \Delta W_i.$$

**Milstein** (strong order 1.0): adds the Itô–Taylor correction $\frac12\sigma^2 S\bigl((\Delta W)^2-h\bigr)$:
$$S_{i+1}=S_i + rS_i h + \sigma S_i \Delta W_i + \tfrac12\sigma^2 S_i\bigl(\Delta W_i^2-h\bigr).$$

Higher strong order means **faster decay of level-difference variance** $V_l$ in MLMC (typically $V_l=O(h_l^2)$ for Milstein vs $O(h_l)$ for Euler on Lipschitz payoffs), which **reduces** the number of samples needed on fine levels and can improve overall complexity from $O(\varepsilon^{-2}(\log\varepsilon)^2)$ to $O(\varepsilon^{-2})$.

Given path values $S_{t_0},\dots,S_{t_n}$ and discount factor $e^{-rT}$:

- Asian call:
  $$P_{\text{Asian}} = e^{-rT}\max\!\left(\frac1{n+1}\sum_{k=0}^n S_{t_k}-K,0\right),$$
- Lookback call (with Brownian interpolation for the running minimum, following Giles and Waterhouse 2009 Eq. 6.1):
  $$P_{\text{Lookback}} = e^{-rT}\max\!\left(S_T-\min_{0<t<T} \hat S(t),0\right),$$
- Digital call:
  $$P_{\text{Digital}} = e^{-rT}\mathbf{1}_{\{S_T>K\}},$$
- Down-and-out barrier call (discrete monitoring):
  $$P_{\text{Barrier}} = e^{-rT}\max(S_T-K,0)\,\mathbf{1}_{\{\min_k S_{t_k}>B\}}.$$

In [2]:
def euler_gbm_path_full_from_dW(S0, r, sigma, T, dW):
    n = len(dW)
    h = T / n
    S = np.zeros(n + 1)
    S[0] = S0
    for i in range(n):
        S[i + 1] = S[i] + r * S[i] * h + sigma * S[i] * dW[i]
    return S

def milstein_gbm_path_full_from_dW(S0, r, sigma, T, dW):
    n = len(dW)
    h = T / n
    S = np.zeros(n + 1)
    S[0] = S0
    for i in range(n):
        S[i + 1] = S[i] + r * S[i] * h + sigma * S[i] * dW[i] + 0.5 * sigma**2 * S[i] * (dW[i]**2 - h)
    return S

def gbm_path_full_from_dW(S0, r, sigma, T, dW, scheme='euler'):
    if scheme == 'milstein':
        return milstein_gbm_path_full_from_dW(S0, r, sigma, T, dW)
    return euler_gbm_path_full_from_dW(S0, r, sigma, T, dW)

def payoff_from_path(S_path, payoff_type, K=1.0, B=0.8, r=0.05, T=1.0):
    disc = np.exp(-r * T)
    ST = S_path[-1]
    if payoff_type == 'asian':
        return disc * max(np.mean(S_path) - K, 0.0)
    if payoff_type == 'lookback':
        return disc * max(ST - np.min(S_path), 0.0)
    if payoff_type == 'digital':
        return disc * (1.0 if ST > K else 0.0)
    if payoff_type == 'barrier':
        alive = np.min(S_path) > B
        return disc * max(ST - K, 0.0) * (1.0 if alive else 0.0)
    raise ValueError(f'Unknown payoff_type: {payoff_type}')

### Randomized rank-1 lattice, Brownian bridge, and importance sampling

**Lattice construction.** Randomized rank-1 lattice:
$$u_i = \big\{\tfrac{i}{N}z + \Delta\big\},\quad i=0,\dots,N-1,$$
with generating vector $z\in\mathbb{Z}^d$ and random shift $\Delta\sim U([0,1)^d)$.

Then map to Gaussians and Brownian increments via **Brownian bridge** to prioritize low-frequency path components in early dimensions.

**Importance sampling for digital payoffs (likelihood ratio method, MLQMC only).**
The digital payoff $\mathbf{1}_{\{S_T>K\}}$ is a hard discontinuity in QMC space, killing lattice gains. We apply a **change of measure** on the first Brownian bridge dimension (which controls $W(T)$, the primary driver of $S_T$).

Set the optimal IS shift so the tilted distribution centres $S_T$ on the strike:
$$\theta = \frac{\ln(K/S_0)-(r-\tfrac12\sigma^2)T}{\sigma\sqrt{T}}.$$

Replace $Z_1$ with $\tilde Z_1 = Z_1+\theta$ when building the terminal Brownian value $W(T)=\sqrt{T}\,\tilde Z_1$, then weight each payoff by the **Radon–Nikodym derivative**:
$$\mathcal{L} = \exp\!\bigl(-\theta Z_1 - \tfrac12\theta^2\bigr),$$
where $Z_1$ is the *original* (unshifted) standard normal. This keeps the estimator **unbiased** while **smoothing** the effective integrand: the QMC lattice no longer has to "find" a thin strip where the digital pays out; the measure shift places the grid right on the boundary, and $\mathcal{L}$ handles the reweighting.

**Why IS applies to MLQMC but not MLMC here:** The Brownian bridge gives $u_1 \mapsto Z_1 \mapsto W(T)$, a **single** QMC dimension controlling the terminal value. Sequential Euler increments (used by MLMC) spread $W(T)$ across **all** $n$ dimensions, so there is no one dimension to shift cleanly.

In [3]:
def rank1_lattice_points(N, d, z, shift):
    i = np.arange(N, dtype=np.int64)[:, None]
    pts = (i * z[None, :]) / N + shift[None, :]
    return pts - np.floor(pts)

def brownian_bridge_schedule(n):
    schedule = []
    intervals = [(0, n)]
    while intervals:
        l, r = intervals.pop(0)
        if r - l <= 1:
            continue
        m = (l + r) // 2
        schedule.append((m, l, r))
        intervals.append((l, m))
        intervals.append((m, r))
    return schedule

def brownian_bridge_dW_from_u(u, T):
    d = len(u)
    t = np.linspace(0.0, T, d + 1)
    u = np.clip(u, 1e-12, 1.0 - 1e-12)
    z = norm.ppf(u)
    W = np.zeros(d + 1)
    W[d] = np.sqrt(T) * z[0]
    idx = 1
    for m, l, r in brownian_bridge_schedule(d):
        tl, tm, tr = t[l], t[m], t[r]
        mean = ((tr - tm) * W[l] + (tm - tl) * W[r]) / (tr - tl)
        var = (tm - tl) * (tr - tm) / (tr - tl)
        W[m] = mean + np.sqrt(max(var, 0.0)) * z[idx]
        idx += 1
        if idx >= len(z):
            break
    return np.diff(W)

def _interp_path_min(S_path, sigma, h, U):
    """Running minimum via Brownian interpolation (Giles & Waterhouse 2009, Eq. 6.1).
    U[i] ~ Uniform(0,1) is the random variable for timestep i."""
    path_min = S_path[0]
    for i in range(len(U)):
        b = sigma * S_path[i]
        arg = (S_path[i+1] - S_path[i])**2 - 2 * b**2 * h * np.log(max(U[i], 1e-300))
        s_min_i = 0.5 * (S_path[i] + S_path[i+1] - np.sqrt(max(arg, 0.0)))
        path_min = min(path_min, s_min_i)
    return path_min

def coupled_payoff_diff_from_dW(dW_fine, payoff_type, S0, r, sigma, T, K, B, M, scheme='euler'):
    S_f = gbm_path_full_from_dW(S0, r, sigma, T, dW_fine, scheme)
    n_f = len(dW_fine)
    h_f = T / n_f

    if payoff_type == 'lookback':
        disc = np.exp(-r * T)
        U = np.random.random(n_f)
        min_f = _interp_path_min(S_f, sigma, h_f, U)
        P_f = disc * max(S_f[-1] - min_f, 0.0)
        if n_f == 1:
            return P_f, P_f
        dW_c = dW_fine.reshape(-1, M).sum(axis=1)
        S_c = gbm_path_full_from_dW(S0, r, sigma, T, dW_c, scheme)
        n_c = len(dW_c)
        W = np.zeros(n_f + 1)
        W[1:] = np.cumsum(dW_fine)
        min_c = S_c[0]
        for j in range(n_c):
            b_c = sigma * S_c[j]
            S_sub = np.zeros(M + 1)
            S_sub[0], S_sub[M] = S_c[j], S_c[j + 1]
            W0, WM = W[j * M], W[(j + 1) * M]
            for k in range(1, M):
                lam = k / M
                S_sub[k] = S_c[j] + lam * (S_c[j+1] - S_c[j]) + b_c * (W[j*M + k] - W0 - lam * (WM - W0))
            for k in range(M):
                arg = (S_sub[k+1] - S_sub[k])**2 - 2 * b_c**2 * h_f * np.log(max(U[j*M + k], 1e-300))
                s_min_k = 0.5 * (S_sub[k] + S_sub[k+1] - np.sqrt(max(arg, 0.0)))
                min_c = min(min_c, s_min_k)
        P_c = disc * max(S_c[-1] - min_c, 0.0)
        return P_f - P_c, P_f

    P_f = payoff_from_path(S_f, payoff_type, K=K, B=B, r=r, T=T)
    if n_f == 1:
        return P_f, P_f
    dW_c = dW_fine.reshape(-1, M).sum(axis=1)
    S_c = gbm_path_full_from_dW(S0, r, sigma, T, dW_c, scheme)
    P_c = payoff_from_path(S_c, payoff_type, K=K, B=B, r=r, T=T)
    return P_f - P_c, P_f

# ── Importance-sampling helpers for digital payoff (likelihood ratio / change of measure) ──

def is_theta_digital(S0, r, sigma, T, K):
    """Optimal IS shift: centres the distribution of S_T on the strike K."""
    return (np.log(K / S0) - (r - 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))

def brownian_bridge_dW_from_u_IS(u, T, theta):
    """Like brownian_bridge_dW_from_u, but shifts the first dimension by theta.
    Returns (dW, z1_original) where z1_original is the *unshifted* standard normal
    (needed for the likelihood ratio weight)."""
    d = len(u)
    t = np.linspace(0.0, T, d + 1)
    u = np.clip(u, 1e-12, 1.0 - 1e-12)
    z = norm.ppf(u)
    z1_original = z[0]
    z[0] = z[0] + theta
    W = np.zeros(d + 1)
    W[d] = np.sqrt(T) * z[0]
    idx = 1
    for m, l, r in brownian_bridge_schedule(d):
        tl, tm, tr = t[l], t[m], t[r]
        mean = ((tr - tm) * W[l] + (tm - tl) * W[r]) / (tr - tl)
        var = (tm - tl) * (tr - tm) / (tr - tl)
        W[m] = mean + np.sqrt(max(var, 0.0)) * z[idx]
        idx += 1
        if idx >= len(z):
            break
    return np.diff(W), z1_original

def coupled_payoff_diff_from_dW_IS(dW_fine, z1, theta, payoff_type, S0, r, sigma, T, K, B, M, scheme='euler'):
    """Coupled payoff with likelihood ratio weight for IS-shifted paths."""
    lr_weight = np.exp(-theta * z1 - 0.5 * theta**2)
    S_f = gbm_path_full_from_dW(S0, r, sigma, T, dW_fine, scheme)
    P_f = payoff_from_path(S_f, payoff_type, K=K, B=B, r=r, T=T) * lr_weight
    if len(dW_fine) == 1:
        return P_f, P_f
    dW_c = dW_fine.reshape(-1, M).sum(axis=1)
    S_c = gbm_path_full_from_dW(S0, r, sigma, T, dW_c, scheme)
    P_c = payoff_from_path(S_c, payoff_type, K=K, B=B, r=r, T=T) * lr_weight
    return P_f - P_c, P_f

### Level estimators and adaptive schemes

For each level:
$$Y_l = \mathbb{E}[\hat P_l-\hat P_{l-1}],\; l\ge1,\qquad Y_0=\mathbb{E}[\hat P_0].$$

- MLMC: i.i.d. coupled paths.
- MLQMC: average over lattice points for each random shift, then average shifts.

We compare complexity via normalized cost $\varepsilon^2\times\text{cost}$.

**Note:** For **discontinuous** payoffs (digital), estimated level variances are noisy and the adaptive loop may require many outer passes before the variance target is met.

In [4]:
def mlmc_level_estimate(level, N, payoff_type, S0, r, sigma, T, K, B, M, scheme='euler'):
    """MLMC level estimator (standard MC, no IS)."""
    diffs = np.zeros(N)
    fine = np.zeros(N)
    n_fine = M ** level
    h = T / n_fine
    for i in range(N):
        dW = np.sqrt(h) * np.random.standard_normal(n_fine)
        val, pf = coupled_payoff_diff_from_dW(dW, payoff_type, S0, r, sigma, T, K, B, M, scheme=scheme)
        diffs[i] = val
        fine[i] = pf
    if level == 0:
        return np.mean(fine), np.var(fine, ddof=1) / N, np.var(fine, ddof=1), N
    return np.mean(diffs), np.var(diffs, ddof=1) / N, np.var(diffs, ddof=1), N * (M**level + M**(level-1))

def mlqmc_level_estimate(level, N_lattice, q_shifts, payoff_type, S0, r, sigma, T, K, B, M, scheme='euler'):
    """MLQMC level estimator. For digital payoffs at level 0, uses IS-shifted Brownian bridge
    to smooth the base payoff. At levels >= 1, IS is *not* applied: the measure shift centres mass
    on the strike boundary, which *increases* the rate of fine/coarse disagreements and can
    raise the variance of P_f - P_c."""
    use_is = (payoff_type == 'digital') and (level == 0)
    theta = is_theta_digital(S0, r, sigma, T, K) if use_is else 0.0
    d = M ** level
    z_gen = 2 * np.arange(1, d + 1, dtype=np.int64) - 1
    y_shift = np.zeros(q_shifts)
    for j in range(q_shifts):
        shift = np.random.random(d)
        U = rank1_lattice_points(N_lattice, d, z_gen, shift)
        vals = np.zeros(N_lattice)
        for i in range(N_lattice):
            if use_is:
                dW_f, z1 = brownian_bridge_dW_from_u_IS(U[i], T, theta)
                val, pf = coupled_payoff_diff_from_dW_IS(dW_f, z1, theta, payoff_type, S0, r, sigma, T, K, B, M, scheme=scheme)
            else:
                dW_f = brownian_bridge_dW_from_u(U[i], T)
                val, pf = coupled_payoff_diff_from_dW(dW_f, payoff_type, S0, r, sigma, T, K, B, M, scheme=scheme)
            vals[i] = pf if level == 0 else val
        y_shift[j] = np.mean(vals)
    Yhat = np.mean(y_shift)
    Vhat = np.var(y_shift, ddof=1) / q_shifts if q_shifts > 1 else 0.0
    Vshift = np.var(y_shift, ddof=1) if q_shifts > 1 else 0.0
    cost = q_shifts * N_lattice * (1 if level == 0 else (M**level + M**(level-1)))
    return Yhat, Vhat, Vshift, cost

def run_experiment_mlmc(
    payoff_type,
    eps,
    S0=S0_ex,
    r=r_ex,
    sigma=sigma_ex,
    T=T_ex,
    K=K_ex,
    B=B_ex,
    M=M_ex,
    N_pilot=1200,
    L_max=None,
    verbose=False,
    scheme='euler',
):
    """Adaptive MLMC with selectable discretisation (euler or milstein).
    If L_max is None, levels increase until the variance and bias criteria are met."""
    L = 0
    Y, Vsingle, N = {}, {}, {}
    var_target = eps**2 / 2
    pass_idx = 0
    tiny = np.finfo(float).tiny
    while True:
        pass_idx += 1
        if L not in Y:
            y, _, vs, _ = mlmc_level_estimate(L, N_pilot, payoff_type, S0, r, sigma, T, K, B, M, scheme=scheme)
            Y[L], Vsingle[L] = y, max(vs, 1e-14)
        sum_sqrt = max(
            sum(np.sqrt(max(Vsingle[l], tiny) * (T / M**l)) for l in range(L + 1)),
            tiny,
        )
        for l in range(L + 1):
            hl = T / (M ** l)
            va = max(Vsingle[l], tiny)
            raw = int(np.ceil(2 * (eps**-2) * np.sqrt(va * hl) / sum_sqrt))
            N[l] = max(raw, 1)
        total_var, total_cost = 0.0, 0
        for l in range(L + 1):
            y, ve, vs, c = mlmc_level_estimate(l, N[l], payoff_type, S0, r, sigma, T, K, B, M, scheme=scheme)
            Y[l], Vsingle[l] = y, max(vs, 1e-14)
            total_var += ve
            total_cost += c
        bias_proxy = abs(Y[L]) if L == 0 else max(abs(Y[L]), abs(Y[L-1]) / M)
        if verbose:
            print(
                f"  MLMC pass {pass_idx}: L={L}, total_var={total_var:.3e} (target {var_target:.3e}), "
                f"bias_proxy={bias_proxy:.3e}, max_N_level={max(N.values())}"
            )
        if total_var <= var_target and (
            (L >= 2 and bias_proxy <= eps / np.sqrt(2)) or (L_max is not None and L >= L_max)
        ):
            break
        if L_max is not None and L >= L_max:
            break
        L += 1
    return {
        'estimate': sum(Y[l] for l in range(L + 1)),
        'cost': total_cost,
        'L': L,
        'bias_proxy': bias_proxy,
        'mlmc_passes': pass_idx,
        'variance_ok': total_var <= var_target,
        'total_var': total_var,
        'scheme': scheme,
    }

def run_experiment_mlqmc(payoff_type, eps, S0=S0_ex, r=r_ex, sigma=sigma_ex, T=T_ex, K=K_ex, B=B_ex, M=M_ex, q_shifts=12, N0=8, L_max=None, scheme='euler'):
    """Adaptive MLQMC with selectable discretisation (euler or milstein).
    If L_max is None, the level count is limited only by the bias stopping criterion."""
    L = 0
    Y, Vest, Nlat = {}, {}, {}
    while True:
        for l in range(L + 1):
            Nlat.setdefault(l, N0)
            y, ve, _, _ = mlqmc_level_estimate(l, Nlat[l], q_shifts, payoff_type, S0, r, sigma, T, K, B, M, scheme=scheme)
            Y[l], Vest[l] = y, max(ve, 1e-16)
        while sum(Vest[l] for l in range(L + 1)) > eps**2 / 2:
            l_star = max(range(L + 1), key=lambda l: Vest[l] / ((M**l) * Nlat[l] + 1e-16))
            Nlat[l_star] *= 2
            y, ve, _, _ = mlqmc_level_estimate(l_star, Nlat[l_star], q_shifts, payoff_type, S0, r, sigma, T, K, B, M, scheme=scheme)
            Y[l_star], Vest[l_star] = y, max(ve, 1e-16)
        bias_proxy = abs(Y[L]) if L == 0 else max(abs(Y[L]), abs(Y[L-1]) / M)
        if L >= 2 and bias_proxy <= eps / np.sqrt(2):
            break
        if L_max is not None and L >= L_max:
            break
        L += 1
    total_cost = sum(
        q_shifts * Nlat[l] * (1 if l == 0 else (M**l + M**(l-1)))
        for l in range(L + 1)
    )
    return {'estimate': sum(Y[l] for l in range(L + 1)), 'cost': total_cost, 'L': L, 'bias_proxy': bias_proxy, 'scheme': scheme}

### Run Section 6 comparison and save plot

We compare **MLMC** and **MLQMC** on `{'asian','lookback','digital','barrier'}`, using the **Milstein** scheme throughout.  The Milstein correction $\frac12\sigma^2 S(\Delta W^2-h)$ raises the strong order from $\frac12$ (Euler) to 1, accelerating the decay of level-difference variance from $V_l=O(h_l)$ to $V_l=O(h_l^2)$ for Lipschitz payoffs and improving MLMC cost from $O(\varepsilon^{-2}(\log\varepsilon)^2)$ to $O(\varepsilon^{-2})$.  For GBM the Itô–Taylor correction has a closed form, so the extra cost per timestep is negligible.

across a **geometric** grid of target accuracies `eps_grid_ex`.

The figure plots $\varepsilon^2\times\text{cost}$ on a log-log scale, with fitted empirical slopes and theoretical reference lines ($O(\varepsilon^{-2})$ flat for MLMC, slope $+1$ for MLQMC at $O(\varepsilon^{-1})$).

In [5]:
payoffs_ex = ['asian', 'lookback', 'digital', 'barrier']

def scheme_for(payoff):
    return 'milstein'

methods = [
    ('MLMC',  lambda p, e: run_experiment_mlmc(p, e, scheme=scheme_for(p)),  'o-',  'C0'),
    ('MLQMC', lambda p, e: run_experiment_mlqmc(p, e, scheme=scheme_for(p)), 's-',  'C2'),
]
results_ex = {p: {m[0]: [] for m in methods} for p in payoffs_ex}

for p in payoffs_ex:
    print(f'--- Payoff: {p} (scheme={scheme_for(p)}) ---')
    for eps in eps_grid_ex:
        parts = []
        for mname, mfn, _, _ in methods:
            out = mfn(p, eps)
            results_ex[p][mname].append((eps, out))
            e2c = eps**2 * out['cost']
            extra = ''
            if 'mlmc_passes' in out:
                extra = f", passes={out['mlmc_passes']}, var_ok={out['variance_ok']}"
            parts.append(f"{mname}: est={out['estimate']:.6f}, L={out['L']}, e2c={e2c:.3f}{extra}")
        print(f"  eps={eps:.4g} | " + " | ".join(parts))

# ── Annotated complexity figure ──
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D

TITLE_MAP = {'asian': 'Asian', 'lookback': 'Lookback', 'digital': 'Digital', 'barrier': 'Barrier'}

shared_xticks = [0.002, 0.003, 0.005, 0.007, 0.01, 0.02]
shared_xticklabels = ['0.002', '0.003', '0.005', '0.007', '0.01', '0.02']

fig, axs = plt.subplots(2, 2, figsize=(12, 9))
axs = axs.ravel()

for ax, p in zip(axs, payoffs_ex):
    eps_arr = np.array([x[0] for x in results_ex[p]['MLMC']])

    for mname, _, style, color in methods:
        cost_arr = np.array([x[1]['cost'] for x in results_ex[p][mname]], dtype=float)
        y = np.maximum(eps_arr**2 * cost_arr, np.finfo(float).tiny)
        ax.loglog(eps_arr, y, style, color=color, markersize=7, linewidth=2.0)

    e_ref = np.array([eps_arr.min() * 0.85, eps_arr.max() * 1.15])

    mlmc_e2c = eps_arr**2 * np.array([x[1]['cost'] for x in results_ex[p]['MLMC']], dtype=float)
    anchor_flat = np.exp(np.mean(np.log(np.maximum(mlmc_e2c, 1e-20))))
    ax.loglog(e_ref, [anchor_flat, anchor_flat], '--', color='#aaaaaa', linewidth=1.3, zorder=1)

    qmc_e2c = eps_arr**2 * np.array([x[1]['cost'] for x in results_ex[p]['MLQMC']], dtype=float)
    log_mid = np.mean(np.log10(eps_arr))
    anchor_slope = np.exp(np.mean(np.log(np.maximum(qmc_e2c, 1e-20))))
    ref_y = anchor_slope * (e_ref / 10**log_mid)
    ax.loglog(e_ref, ref_y, '--', color='#aaaaaa', linewidth=1.3, zorder=1)

    ax.set_title(TITLE_MAP[p], fontsize=14, fontweight='bold', pad=8)
    ax.grid(True, which='major', alpha=0.3, linewidth=0.6)
    ax.grid(True, which='minor', alpha=0.12, linewidth=0.4)

    ax.set_xticks(shared_xticks)
    ax.set_xticklabels(shared_xticklabels)
    ax.xaxis.set_minor_formatter(mticker.NullFormatter())
    ax.set_xlim(0.0015, 0.028)
    ax.set_ylim(5e-3, 50)
    ax.tick_params(axis='both', which='major', labelsize=9.5, width=1.0, length=5)
    ax.tick_params(axis='both', which='minor', width=0.6, length=3)

    ax.set_xlabel(r'$\varepsilon$', fontsize=12)
    if p in ('asian', 'digital'):
        ax.set_ylabel(r'$\varepsilon^2 \times \mathrm{cost}$', fontsize=12)

legend_handles = [
    Line2D([0], [0], color='C0', marker='o', linewidth=2.0, markersize=7, label='MLMC'),
    Line2D([0], [0], color='C2', marker='s', linewidth=2.0, markersize=7, label='MLQMC'),
    Line2D([0], [0], color='#aaaaaa', linewidth=1.3, linestyle='--', label=r'$O(\varepsilon^{-2})$ reference (flat)'),
    Line2D([0], [0], color='#aaaaaa', linewidth=1.3, linestyle='--',
           marker=r'$\nearrow$', markersize=8, label=r'$O(\varepsilon^{-1})$ reference (slope 1)'),
]
fig.legend(handles=legend_handles, loc='lower center', ncol=4, fontsize=10,
           frameon=True, framealpha=0.95, edgecolor='#cccccc',
           bbox_to_anchor=(0.5, -0.01))

plt.tight_layout(rect=[0, 0.045, 1, 1], h_pad=2.5, w_pad=2.0)
exotic_cost_path = os.path.join(IMAGES_DIR, 'exotic_mlmc_mlqmc_comparison.png')
fig.savefig(exotic_cost_path, dpi=300, bbox_inches='tight')
print(f"Saved: {exotic_cost_path}")
plt.show()

--- Payoff: asian (scheme=milstein) ---
  eps=0.02 | MLMC: est=0.056529, L=2, e2c=4.727, passes=3, var_ok=True | MLQMC: est=0.058762, L=2, e2c=0.998
  eps=0.01439 | MLMC: est=0.057434, L=2, e2c=4.725, passes=3, var_ok=True | MLQMC: est=0.064509, L=2, e2c=0.517
  eps=0.01036 | MLMC: est=0.057080, L=2, e2c=4.789, passes=3, var_ok=True | MLQMC: est=0.051826, L=2, e2c=0.268
  eps=0.007455 | MLMC: est=0.055878, L=2, e2c=4.688, passes=3, var_ok=True | MLQMC: est=0.058931, L=2, e2c=0.139
  eps=0.005365 | MLMC: est=0.055951, L=2, e2c=4.765, passes=3, var_ok=True | MLQMC: est=0.053251, L=2, e2c=0.075
  eps=0.003861 | MLMC: est=0.056646, L=2, e2c=4.778, passes=3, var_ok=True | MLQMC: est=0.058998, L=2, e2c=0.049
  eps=0.002779 | MLMC: est=0.057262, L=3, e2c=5.533, passes=4, var_ok=True | MLQMC: est=0.055251, L=2, e2c=0.033
  eps=0.002 | MLMC: est=0.057538, L=3, e2c=5.490, passes=4, var_ok=True | MLQMC: est=0.059809, L=3, e2c=0.049
--- Payoff: lookback (scheme=milstein) ---
  eps=0.02 | MLMC: est

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/numpy/_core/fromnumeric.py:4268: RuntimeWarning: Degrees of freedom <= 0 for slice
  return _methods._var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/numpy/_core/_methods.py:214: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


ValueError: cannot convert float NaN to integer